[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Imágenes de contenedor y registro.

Un **contenedor** empaqueta una aplicación junto con su entorno de ejecución, de modo que el mismo artefacto pueda correr en el *Codespace*, en un *pipeline* y en el destino de despliegue. La receta de ese empaquetado es un archivo llamado ```Dockerfile```. El resultado de construirla es una **imagen**; un **registro** es el almacén desde el que se publica y se descarga esa imagen.

## Objetivos.

* Leer el ```Dockerfile``` de la raíz del repositorio y relacionar cada instrucción con el arranque de la *API* del cuaderno 14.
* Construir y ejecutar la imagen en el *Codespace*, y comprobar ```GET /health```.
* Distinguir la construcción de comprobación del *job* ```verificar```, la publicación en *GitHub Container Registry* (*GHCR*, el registro de contenedores de *GitHub*) y la promoción de esa misma imagen (misma etiqueta *SHA*) hacia *Artifact Registry* y hacia *Linux*.

## El ```Dockerfile```.

El ```Dockerfile``` (y el ```.dockerignore```) están en la **raíz** de este repositorio y construyen la *API* del directorio ```api/```.

La siguiente celda mostrará el contenido vigente de ese ```Dockerfile```.

```dockerfile
# syntax=docker/dockerfile:1
# Imagen de la API de cd441 (Cloudevel). Se construye desde la RAÍZ del repositorio:
#   docker build -t cd441-api .
#
# Buenas prácticas que muestra este archivo:
#  1. Base con versión fija Y digest (no `latest`): la misma entrada da la misma
#     imagen. Dependabot (.github/dependabot.yml) propone actualizar el digest.
#  2. Dependencias antes que el código: aprovecha la caché de capas.
#  3. Solo se copia lo necesario para ejecutar (ni pruebas, ni .env, ni deploy/).
#  4. El código pertenece a root y el proceso corre con un usuario sin privilegios:
#     la aplicación no puede modificarse a sí misma.
FROM python:3.12-slim-trixie@sha256:191083cdf6c7bda8c76158983163ead5d5cf7b77cf3033e805bb0f5296f51b4b

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    PIP_DISABLE_PIP_VERSION_CHECK=1 \
    PORT=8080

WORKDIR /app

# Usuario sin privilegios con UID fijo (el mismo en Docker, Compose y Cloud Run).
RUN groupadd --system --gid 10001 appuser \
    && useradd --system --uid 10001 --gid appuser --no-create-home --shell /usr/sbin/nologin appuser

COPY api/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY api/app/ /app/app/
COPY api/wsgi.py /app/wsgi.py

# Numérico (10001 = appuser) para que plataformas como Kubernetes verifiquen
# que no es root sin consultar /etc/passwd.
USER 10001:10001

EXPOSE 8080

# Útil en Compose o en un servidor Linux. Cloud Run ignora esta instrucción
# y hace su propia comprobación de arranque.
HEALTHCHECK --interval=15s --timeout=5s --start-period=10s --retries=3 \
    CMD ["python", "-c", "import os, urllib.request; urllib.request.urlopen('http://127.0.0.1:%s/health' % os.environ.get('PORT', '8080'), timeout=4)"]

# gunicorn en primer plano, un solo worker (el almacén en memoria es por proceso).
# Cloud Run inyecta PORT. --timeout acotado: una petición colgada no vive para siempre.
CMD ["sh", "-c", "exec gunicorn --bind 0.0.0.0:${PORT} --workers 1 --threads 4 --timeout 30 --access-logfile - wsgi:app"]
```

Puntos que conviene observar:

* La imagen base lleva una **versión fija y un *digest*** (```python:3.12-slim-trixie@sha256:…```), no la etiqueta ```latest```. *Dependabot* propone actualizar ese *digest*.
* Las dependencias de ejecución salen de ```api/requirements.txt``` (no de ```requirements-dev.txt```, que es solo para linter y pruebas).
* Solo se copian ```api/app/``` y ```api/wsgi.py```: ni pruebas, ni ```.env```, ni ```api/deploy/```.
* El código pertenece a *root* y el proceso corre como el usuario ```10001```, sin privilegios. La aplicación no puede modificarse a sí misma.
* El arranque es *gunicorn* sobre ```wsgi:app```, en primer plano, con un solo *worker* y un ```--timeout``` acotado.
* El puerto se configura con la variable de entorno ```PORT```, que *Cloud Run* también inyecta.
* Hay un ```HEALTHCHECK``` sobre ```GET /health``` (útil en *Compose* o en un servidor *Linux*; *Cloud Run* hace su propia comprobación).
* El ```.dockerignore``` evita copiar entornos virtuales, bitácoras, cachés y secretos a la imagen.

## Construcción en el *Codespace*.

En la raíz del repositorio, dentro del *Codespace* (donde *Docker* ya está disponible):

La siguiente celda construye la imagen ```cd441-api``` a partir del ```Dockerfile``` de la raíz. La primera vez descarga la imagen base; en construcciones posteriores, *Docker* reutiliza las capas que no cambiaron (en la salida aparecen como ```CACHED```).

In [ ]:
docker build -t cd441-api .

La siguiente celda ejecuta la imagen en segundo plano (```-d```) con el nombre ```cd441-api```, publica el puerto 8080 y define ```APP_VERSION=local```. El cuaderno queda libre para la comprobación; ```docker ps``` debe mostrar el contenedor con estado ```Up```.

In [ ]:
docker run -d --rm --name cd441-api -p 8080:8080 -e APP_VERSION=local cd441-api
sleep 3
docker ps --filter name=cd441-api

La clave de *Flask* y las demás variables de entorno se documentan en ```api/.env.example```. En un despliegue se definen en el entorno del contenedor; no se incrustan en la imagen ni se versionan valores reales. Como no hay ```DATABASE_URL```, el almacén es el de memoria.

La comprobación es una petición *HTTP* (*Hypertext Transfer Protocol*) a la ruta de salud:

La siguiente celda comprobará la ruta de salud con una petición *HTTP*.

In [ ]:
curl -s http://127.0.0.1:8080/health

**Salida típica** (almacén en memoria, sin base de datos):

```json
{"almacen":"memoria","status":"ok","version":"local"}
```

Si hubiera base configurada y no respondiera, la misma ruta devolvería 503 con ```"status": "error"```. La etiqueta ```cd441-api``` es solo de trabajo en el *Codespace*. En el *pipeline* se usa una etiqueta ligada al *SHA* (*Secure Hash Algorithm*) del *commit* y un registro compartido.

La siguiente celda detiene el contenedor. Como se lanzó con ```--rm```, *Docker* lo elimina al detenerlo; la imagen ```cd441-api``` sigue disponible.

In [ ]:
docker stop cd441-api

## Dónde se construye, dónde se publica y dónde se promociona.

El *job* ```verificar``` de ```.github/workflows/ci.yml``` incluye un paso ```docker build -t cd441-api:ci .``` **sin** ```docker push```. Su función es fallar pronto si el ```Dockerfile``` está roto.

El *job* ```publicar-imagen``` (solo tras un *CI* verde en un *push* a ```main```) publica esa imagen en *GHCR*, con la etiqueta igual al *SHA* del *commit* (nunca solo ```latest```). El paquete se deja **público** (*Package settings* → *Change visibility*), para que el anfitrión *Linux* y *Cloud Run* (vía *Artifact Registry*) puedan descargarlo sin credenciales adicionales. El *job* pide ```packages: write``` solo ahí, y se autentica en *GHCR* con el ```GITHUB_TOKEN``` temporal, no con un *PAT* (*personal access token*) de larga duración.

La regla de este curso es **construir una vez y promocionar**: no se reconstruye la imagen por entorno. El cuaderno 16 copia la imagen de *GHCR* al *Artifact Registry* de *Google Cloud Platform* (*GCP*) con la misma etiqueta *SHA*. El cuaderno 17 la descarga de *GHCR* en el servidor *Linux*.

En el destino *Linux*, el manifiesto ```api/deploy/compose.yaml``` refuerza otras prácticas de contenedor: sistema de archivos de solo lectura, ```tmpfs```, ```cap_drop: [ALL]``` y ```no-new-privileges```.

## Actividad.

1. Construir la imagen en el *Codespace* y comprobar que ```GET /health``` responde con ```"almacen": "memoria"``` y ```"status": "ok"```.
2. Revisar en una ejecución del *workflow* ```CI``` el paso ```Construir imagen (sin publicar)``` del *job* ```verificar``` y, en un *push* a ```main```, el *job* ```publicar-imagen```: la etiqueta debe ser el *SHA* del *commit*, no ```latest```.
3. Abrir el ```Dockerfile``` y señalar qué instrucción impide que el proceso corra como *root*, cuál fija el *digest* de la imagen base y cuál declara el ```HEALTHCHECK```.

El cuaderno 16 promociona esa misma imagen de *GHCR* a *Artifact Registry* y la despliega en *Cloud Run*.

## Cierre.

La *API* ya no es solo un proyecto de *Python*: es una imagen con base fijada por versión y *digest*, usuario sin privilegios, código no modificable por el proceso y un arranque explícito. Esa imagen se publica una vez en *GHCR* y se promociona, sin reconstruirla por entorno, en *Cloud Run* y en *Linux*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>